# Assignment 2 - Road Collision Analysis
This notebook cleans, prepares and analyeses the Uk road collision and vehicle dates using panda and numpy  

# 1. Load and Inspect

In [1]:
# import libraries 
import pandas as pd 
import numpy as np 

In [2]:
# load datasets from the datasets folder 
collision_df = pd.read_csv("datasets/road_collision_2023.csv")
vehicle_df = pd.read_csv("datasets/road_vehicle_2023.csv")

In [3]:
#display datasets shapes
print("collision datasets shape:", collision_df.shape)
print("vehicle datasets shape:" , vehicle_df.shape)

collision datasets shape: (8160, 44)
vehicle datasets shape: (14558, 32)


In [4]:
# display datasets data types for collision dataset

print("collision dataset data types:")
display(collision_df.dtypes)

#display datasets data type for vehicle dataset

print("\nvehicle dataset data types:")
display(vehicle_df.dtypes)

collision dataset data types:


collision_index                                         str
collision_year                                        int64
collision_ref_no                                        str
location_easting_osgr                               float64
location_northing_osgr                              float64
longitude                                           float64
latitude                                            float64
police_force                                          int64
collision_severity                                    int64
number_of_vehicles                                  float64
number_of_casualties                                float64
date                                                    str
day_of_week                                           int64
time                                                    str
local_authority_district                              int64
local_authority_ons_district                            str
local_authority_highway                 


vehicle dataset data types:


collision_index                                str
collision_year                               int64
collision_ref_no                               str
vehicle_reference                            int64
vehicle_type                                 int64
towing_and_articulation                      int64
vehicle_manoeuvre_historic                   int64
vehicle_manoeuvre                            int64
vehicle_direction_from                       int64
vehicle_direction_to                         int64
vehicle_location_restricted_lane_historic    int64
vehicle_location_restricted_lane             int64
junction_location                            int64
skidding_and_overturning                     int64
hit_object_in_carriageway                    int64
vehicle_leaving_carriageway                  int64
hit_object_off_carriageway                   int64
first_point_of_impact                        int64
vehicle_left_hand_drive                      int64
journey_purpose_of_driver_histo

In [5]:
#display first 3 rows 
display(collision_df.head(3))
display(vehicle_df.head(3))

,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
0,2023545293223,2023,545293223,414313.0,189982.0,-1.794709,51.608494,54,2,1.0,...,0,0,2,1,2,E01015471,-1,0,1.000000,0.000000
1,2023542622123,2023,542622123,411924.0,131455.0,-1.831150,51.082294,54,3,2.0,...,0,0,1,1,2,E01031981,-1,0,0.016403,0.983597
2,2023340NN1932,2023,340NN1932,486125.0,278980.0,-0.735522,52.402067,34,3,2.0,...,0,0,1,3,2,E01027104,-1,0,0.006125,0.993875


,collision_index,collision_year,collision_ref_no,vehicle_reference,vehicle_type,towing_and_articulation,vehicle_manoeuvre_historic,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,...,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding
0,2023461304428,2023,461304428,1,9,0,18,19,8,4,...,37,7,2979,1,8,BMW M4,4,E01024528,0,-1
1,2023451334257,2023,451334257,2,9,0,12,12,5,1,...,35,6,2979,1,7,BMW M4,8,E01001022,0,-1
2,2023131282212,2023,131282212,2,9,0,18,19,1,5,...,53,8,2979,1,5,BMW M4,10,E01011198,0,-1


In [6]:
#display missing values for collision datasets 

print("missing values in collision datasets:")
print(collision_df.isna().sum())

#display missing values for vehicle dataset

print("missing values in vechile dataset")
print(vehicle_df.isna().sum())

missing values in collision datasets:
collision_index                                      0
collision_year                                       0
collision_ref_no                                     0
location_easting_osgr                                1
location_northing_osgr                               1
longitude                                            1
latitude                                             1
police_force                                         0
collision_severity                                   0
number_of_vehicles                                  41
number_of_casualties                                41
date                                                 0
day_of_week                                          0
time                                                 0
local_authority_district                             0
local_authority_ons_district                         0
local_authority_highway                              0
local_authority_highway_cur

### Observation for Step 1

The datasets were loaded successfully from the datasets folder.
The collision dataset contains road accident information, while the vehicle dataset contains vehicle-related information.
The data type inspection showed that the datasets contain integer, float and string columns.
The missing value analysis helped identify columns that may need cleaning before further analysis

# 2. Data cleaning 

In [7]:
# check unique values in collision serverity 
print(collision_df['collision_severity'].unique())

[2 3 1 9]


In [8]:
#replacing invalid collision severity values with NaN

collision_df['collision_severity'] = collision_df['collision_severity'].where(
    collision_df['collision_severity'].isin([1,2,3]),
    np.nan
)

In [9]:
#count missing values after cleaning collision severity 
print(collision_df['collision_severity'].isna().sum())

1


In [10]:
#find out duplicate rows  in collision datasat

print("duplicate rows in collision dataset:")
print(collision_df.duplicated().sum())

#find out duplicate rows in vehicle dataset

print("/nduplicate rows in vehicle dataset:")
print(vehicle_df.duplicated().sum())

duplicate rows in collision dataset:
151
/nduplicate rows in vehicle dataset:


0


In [11]:
#remove duplicate rows
collision_df = collision_df.drop_duplicates()

vehicle_df = vehicle_df.drop_duplicates()

In [12]:
# clean invalid driver ages
vehicle_df['age_of_driver'] = vehicle_df['age_of_driver'].replace(-1, np.nan)

# remove ages below 16
vehicle_df.loc[
    vehicle_df['age_of_driver'] < 16,
    'age_of_driver'
] = np.nan

# clean unrealistic speed limits
collision_df['speed_limit'] = collision_df['speed_limit'].replace(
    [0, 5, 150],
    np.nan
)

# clean suspicious casualty values
collision_df['number_of_casualties'] = collision_df[
    'number_of_casualties'
].replace(99, np.nan)


print("Driver ages below 16:", (vehicle_df['age_of_driver'] < 16).sum())
print("Speed limits after cleaning:", sorted(collision_df['speed_limit'].dropna().unique()))
print("Maximum casualties after cleaning:", collision_df['number_of_casualties'].max())

Driver ages below 16: 0
Speed limits after cleaning: [np.float64(20.0), np.float64(30.0), np.float64(40.0), np.float64(50.0), np.float64(60.0), np.float64(70.0)]
Maximum casualties after cleaning: 16.0


### Observation for Step 2


Several data cleaning steps were performed to improve the quality of both datasets before analysis.

Duplicate rows were checked in both datasets. A total of 151 duplicate rows were found and removed from the collision dataset, while no duplicate rows were found in the vehicle dataset.

Invalid values in the age_of_driver column were cleaned. The value -1 was treated as an unknown/sentinel value and replaced with NaN. Driver ages below 16 were also replaced with NaN because they are unrealistic for licensed drivers in the UK.

Unrealistic speed_limit values were cleaned in the collision dataset. Values of 0, 5, and 150 were identified as implausible UK road speed limits and replaced with NaN.

Suspicious values in number_of_casualties were cleaned. The value 99 was treated as an invalid placeholder/error value and replaced with NaN.

These cleaning steps improved the reliability and consistency of the datasets for further analysis.

# 3. Filtering and Basic Summaries

In [13]:
# keep only rows with valid collision severity values

clean_collision_df = collision_df[
    collision_df['collision_severity'].isin([1,2,3])
].copy()

In [14]:
# display top 10 collisions by number of casualties
top_collisions = clean_collision_df[
    [
        'collision_index',
        'police_force',
        'collision_severity',
        'number_of_casualties',
        'number_of_vehicles',
        'speed_limit'
    ]
].sort_values(
    by='number_of_casualties',
    ascending=False
).head(10)

display(top_collisions)

,collision_index,police_force,collision_severity,number_of_casualties,number_of_vehicles,speed_limit
4056,2023061339517,6,2.0,16.0,1.0,30.0
4133,2023061363978,6,2.0,14.0,3.0,30.0
551,2023401265937,40,3.0,9.0,12.0,70.0
6354,2023131375943,13,2.0,8.0,2.0,30.0
5160,2023421312420,42,3.0,7.0,2.0,70.0
7112,2023552400153,55,3.0,7.0,5.0,30.0
1640,2023061341053,6,3.0,7.0,4.0,30.0
6452,2023201312784,20,3.0,6.0,4.0,30.0
1769,2023991283815,99,3.0,6.0,3.0,50.0
4116,2023041315163,4,2.0,6.0,2.0,30.0


In [15]:
#count collision by police force 
police_force_counts = clean_collision_df['police_force'].value_counts()

display(police_force_counts.head(10))

police_force
1     1749
20     372
13     346
99     320
46     299
47     258
43     247
44     225
4      217
50     213
Name: count, dtype: int64

### Observation for Step 3

The filtered dataset now contains only valid collision severity values.
The top collision records showed incidents with a high number of casualties and vehicles involved.
The police force summary showed that Police_force 1 had the highest number of collisions with 1,748 recorded incidents, significantly more than any other force, indicating uneven collision distribution across regions.

# 4. Grouped Analysis

In [16]:
# Creat safty score 

clean_collision_df['safety_score'] = 4 - clean_collision_df['collision_severity']

# group by police_force
force_summary = clean_collision_df.groupby('police_force').agg(
    Avg_safety_score=('safety_score' , 'mean'),
    N_collisions=('collision_index' , 'nunique'),
    Total_casualties=('number_of_casualties' , 'sum')
).round(2).sort_values(
    by='Avg_safety_score',
    ascending=False)

display(force_summary)





,Avg_safety_score,N_collisions,Total_casualties
police_force,,,
99,1.48,320,433.0
11,1.45,40,44.0
21,1.45,62,78.0
14,1.39,158,199.0
63,1.38,74,108.0
60,1.38,50,73.0
33,1.35,109,139.0
6,1.34,197,287.0
13,1.34,346,440.0


In [17]:
# group by collision_year and police_force
year_force_summary = clean_collision_df.groupby(
    ['collision_year', 'police_force']
).agg(
    Avg_safety_score=('safety_score', 'mean'),
    N_collisions=('collision_index', 'nunique')
).round(2)

display(year_force_summary)

Avg_safety_score  N_collisions
collision_year police_force                                
2023           1                         1.16          1748
               3                         1.29            73
               4                         1.29           217
               5                         1.21           145
               6                         1.34           197
               7                         1.19           120
               10                        1.22           146
               11                        1.45            40
               12                        1.31           110
               13                        1.34           346
               14                        1.39           158
               16                        1.29           159
               17                        1.19            42
               20                        1.19           372
               21                        1.45            62
               22                        1.32           113
               23                        1.18            76
               30                        1.24           140
               31                        1.31           129
               32                        1.23           146
               33                        1.35           109
               34                        1.14           112
               35                        1.23           126
               36                        1.31           107
               37                        1.33            73
               40                        1.19           101
               41                        1.23           163
               42                        1.26           204
               43                        1.24           246
               44                        1.21           224
               45                        1.29           200
               46                        1.23           298
               47                        1.26           258
               48                        1.31            16
               50                        1.31           213
               52                        1.22           193
               53                        1.29            70
               54                        1.23            98
               55                        1.26            93
               60                        1.38            50
               61                        1.24            34
               62                        1.21            78
               63                        1.38            74
               99                        1.48           320

## Observation for Grouped Analysis


The grouped analysis shows differences in safety scores across police force areas.Police force 99 had the highest average safety score of 1.48 with 320 collisions and 433, total casualties, followed by police forces 11 and 21 both scoring 1.45.
Police force 34 had the lowest average safety score of 1.14 across 112 collisions and its suggesting a higher proportion of serious collisions in that area.
Police force 1 recorded the most collisions overall (1,748) with a below-average safety score of 1.16, indicating both high volume and higher severity.
The year and police force breakdown track whether these patterns are consistentover time or show regional trends.

## 5. Filters by severity and size

In [18]:
#Identify collisions with severity <= 2 and at least 2 casualties.

severe_collisions = clean_collision_df[
    (clean_collision_df['collision_severity'] <=2) &
    (clean_collision_df['number_of_casualties'] >= 2)
    ]

print("number of collsion:" , len(severe_collisions))

display(severe_collisions.head(5))

number of collsion: 452


,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,safety_score
10,2023131348940,2023,131348940,433169.0,434033.0,-1.497872,53.801548,13,2.0,1.0,...,0,1,1,2,E01011340,5,1,1.0,0.0,2.0
21,2023421296798,2023,421296798,597263.0,225455.0,0.865383,51.892728,42,2.0,3.0,...,0,2,1,2,E01021676,5,1,1.0,0.0,2.0
25,2023332300200,2023,332300200,434650.0,314140.0,-1.488406,52.723806,33,1.0,2.0,...,0,2,1,2,E01025921,-1,0,0.0,0.0,3.0
41,2023052400589,2023,052400589,334559.0,418180.0,-2.991659,53.656002,5,2.0,2.0,...,0,1,1,2,E01006944,-1,0,1.0,0.0,2.0
59,2023320658678,2023,320658678,518387.0,384146.0,-0.223329,53.340977,32,2.0,3.0,...,20,2,1,2,E01026071,-1,0,1.0,0.0,2.0


In [19]:
#Identify collisions with severity = 3 and exactly 1 casualty.

minor_collisions = clean_collision_df[
    (clean_collision_df['collision_severity'] ==3) &
    (clean_collision_df['number_of_casualties'] ==1)
]

print("number of collisions:", len(minor_collisions))

display(minor_collisions.head(5))

number of collisions: 5004


,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,safety_score
1,2023542622123,2023,542622123,411924.0,131455.0,-1.831150,51.082294,54,3.0,2.0,...,0,1,1,2,E01031981,-1,0,0.016403,0.983597,1.0
2,2023340NN1932,2023,340NN1932,486125.0,278980.0,-0.735522,52.402067,34,3.0,2.0,...,0,1,3,2,E01027104,-1,0,0.006125,0.993875,1.0
4,2023061324154,2023,061324154,387235.0,406083.0,-2.194137,53.551222,6,3.0,2.0,...,-1,1,3,2,E01005539,3,1,0.000000,1.000000,1.0
5,2023052301780,2023,052301780,340304.0,385768.0,-2.898559,53.365395,5,3.0,1.0,...,0,1,2,2,E01006536,-1,0,0.192109,0.807891,1.0
7,2023141262816,2023,141262816,439717.0,394947.0,-1.403432,53.449797,14,3.0,2.0,...,0,1,2,2,E01007804,3,1,0.000000,1.000000,1.0


In [20]:
# Identify collisions where collision_adjusted_severity_serious is >= to 60%.

serious_adjusted = clean_collision_df[
    clean_collision_df['collision_adjusted_severity_serious'] >= 0.60
    ]
print("number of collisions:" , len(serious_adjusted))

display(serious_adjusted.head(5))

number of collisions: 1821


,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,safety_score
0,2023545293223,2023,545293223,414313.0,189982.0,-1.794709,51.608494,54,2.0,1.0,...,0,2,1,2,E01015471,-1,0,1.0,0.0,2.0
9,2023201267385,2023,201267385,433432.0,280284.0,-1.509842,52.419534,20,2.0,2.0,...,0,1,2,2,E01009629,6,1,1.0,0.0,2.0
10,2023131348940,2023,131348940,433169.0,434033.0,-1.497872,53.801548,13,2.0,1.0,...,0,1,1,2,E01011340,5,1,1.0,0.0,2.0
12,2023481377458,2023,481377458,532184.0,181047.0,-0.096513,51.512895,48,2.0,2.0,...,0,1,3,2,E01032739,7,1,1.0,0.0,2.0
14,2023010420014,2023,010420014,533228.0,183096.0,-0.080703,51.531063,1,2.0,1.0,...,0,1,1,2,E01001784,-1,0,1.0,0.0,2.0



### Observation for Step 5

The results show different categories of collisions based on severity and casualty counts. 

Collisions with severity less than or equal to 2 and multiple casualties represent more serious accidents that may require greater emergency response and safety measures. 

Collisions with severity equal to 3 and exactly one casualty appear more frequently and may represent less severe incidents. 

The filter using collision_adjusted_severity_serious greater than or equal to 60% helps identify collisions with a high level of seriousness according to the adjusted severity metric. 

These filters help focus analysis on specific types of collisions and support better understanding of accident severity patterns.


## 6. NumPy Analysis

In [21]:
#convert colums to numpy arrays

import numpy as np

#Convert speed_limit and number_of_casualties to NumPy arrays

speed_array = clean_collision_df['speed_limit'].to_numpy()

casualty_array = clean_collision_df['number_of_casualties'].to_numpy()

In [22]:
# Computing the covariance between  two variables using np.cov. keep only rows where both values are finite.

valid_mask = np.isfinite(speed_array) & np.isfinite(casualty_array)

valid_speed = speed_array[valid_mask]

valid_casualties = casualty_array[valid_mask]

# covariance matrix

cov_matrix = np.cov(valid_speed, valid_casualties)

# extract covariance value

covariance_value = cov_matrix [0,1] 

print("covariance between speed limit and casualties:")

print(covariance_value)


covariance between speed limit and casualties:
1.877324486761192


In [23]:
#Computing the 25th, 50th, and 75th percentiles of number_of_casualties using np.percentile. Using only finite values.

finite_casualties = casualty_array[np.isfinite(casualty_array)]

percentiles = np.percentile(finite_casualties, [25,50,75])

print("25th, 50th, and 75th percentiles:")

print(percentiles)

25th, 50th, and 75th percentiles:
[1. 1. 1.]


In [24]:
# Computing the proportion of collisions with at least two casualties using the mean of a boolean array.

two_or_more = finite_casualties >= 2

# propertion using mean 

proportion = np.mean(two_or_more)

print("proportion of collisions with at least 2 casualities:")

print(proportion)

proportion of collisions with at least 2 casualities:
0.18817474265628922


### Observation for Step 6
The covariance was around 1.88 which shows speed and casualties are slightly related. Most collisions only had 1 casualty since all three percentiles came out as 1.0. About 19% of collisions had 2 or more casualties which is quite a lot when you think about it.
These NumPy calculations provide additional statistical insight into collision severity and casualty patterns.

## 7. Derived Column

In [25]:
# creating new column multi_vehicle_collision filtered collision data 

clean_collision_df['multi_vehicle_collision'] = np.where(
    clean_collision_df['number_of_vehicles'].fillna(0) >= 2,
    1,
    0
)

# display sample of 5 rows

sample_rows = clean_collision_df[
    [
        'collision_index',
        'number_of_vehicles',
        'number_of_casualties',
        'multi_vehicle_collision'
    ]
].head(5)

display(sample_rows)

,collision_index,number_of_vehicles,number_of_casualties,multi_vehicle_collision
0,2023545293223,1.0,1.0,0
1,2023542622123,2.0,1.0,1
2,2023340NN1932,2.0,1.0,1
3,2023122301161,2.0,3.0,1
4,2023061324154,2.0,1.0,1


### Observation for Step 7

The new `multi_vehicle_collision` column classifies collisions based on the number of vehicles involved. 
A value of 1 indicates that the collision involved two or more vehicles, while 0 indicates that fewer than two vehicles were involved. Missing values in `number_of_vehicles` were treated as 0 to ensure that every row received a valid classification. 

This derived column helps simplify the identification of multi-vehicle accidents and can support further analysis of collision severity and casualty patterns

## 8. Vehicle Dataset Analysis

In [26]:
#Group by vehicle_type and compute

vehicle_summary = vehicle_df.groupby('vehicle_type').agg(
    mean_age_of_driver=('age_of_driver', 'mean'),
    number_of_rows=('vehicle_type', 'size'),
    unique_collision=('collision_index' , 'nunique')
).round(2)

display(vehicle_summary)



,mean_age_of_driver,number_of_rows,unique_collision
vehicle_type,,,
-1,30.20,119,118
1,39.63,1238,1216
2,27.74,78,78
3,30.25,758,751
4,35.61,144,143
5,44.72,331,323
8,45.85,226,224
9,42.81,9968,6795
10,48.19,18,18


### Observation for Step 8

The vehicle dataset was grouped by vehicle_type to compare driver age, total records, and the number of unique collisions associated with each type of vehicle. 
The mean driver age provides insight into the typical driver profile for different vehicle categories, while the number of rows and unique collisions help show which vehicle types appear most frequently in the dataset. 
Displaying results with two decimal places improves readability and presentation quality.

## 9. Merge and Merged Analysis

In [27]:
#Merging the filtered collision data with the cleaned vehicle dataset using the collision_index column.

# convert age_of_driver to numeric values

vehicle_df['age_of_driver'] = pd.to_numeric(
    vehicle_df['age_of_driver'],
    errors='coerce'
)

In [28]:
# Merge the filtered collision data with the cleaned vehicle dataset

merged_df = pd.merge(
    clean_collision_df,
    vehicle_df,
    on='collision_index',
    how='inner'
)

display(merged_df.head(5))


,collision_index,collision_year_x,collision_ref_no_x,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding
0,2023545293223,2023,545293223,414313.0,189982.0,-1.794709,51.608494,54,2.0,1.0,...,87.0,11,996,1,14,SUZUKI ALTO,8,E01032709,0,-1
1,2023542622123,2023,542622123,411924.0,131455.0,-1.831150,51.082294,54,3.0,2.0,...,39.0,7,2143,2,9,MERCEDES C CLASS,2,E01031981,0,-1
2,2023542622123,2023,542622123,411924.0,131455.0,-1.831150,51.082294,54,3.0,2.0,...,37.0,7,-1,-1,-1,-1,2,E01031981,0,-1
3,2023340NN1932,2023,340NN1932,486125.0,278980.0,-0.735522,52.402067,34,3.0,2.0,...,77.0,11,998,1,17,DAIHATSU SIRION,7,E01027102,0,-1
4,2023340NN1932,2023,340NN1932,486125.0,278980.0,-0.735522,52.402067,34,3.0,2.0,...,NaN,-1,-1,-1,-1,-1,-1,-1,0,-1


In [29]:
# grouped analysis using merged data

merged_summary = merged_df.groupby('police_force').agg(
    Median_age_of_driver=('age_of_driver', 'median'),
    Std_age_of_driver=('age_of_driver', 'std'),
    Number_of_merged_rows=('collision_index', 'size'),
    Distinct_vehicle_types=('vehicle_type', 'nunique')
).round(2).sort_values(
    by='Median_age_of_driver',
    ascending=False
)

display(merged_summary)

,Median_age_of_driver,Std_age_of_driver,Number_of_merged_rows,Distinct_vehicle_types
police_force,,,,
3,46.0,17.28,127,12
36,45.0,18.24,195,15
54,45.0,17.97,182,9
4,42.0,17.31,403,12
42,42.0,18.77,381,14
60,42.0,17.81,95,10
63,41.5,18.70,127,14
17,41.5,16.77,78,8
12,41.5,18.32,188,12


In [30]:
#Creating a pivot table with police_force as rows and vehicle_type as columns

pivot_table = pd.pivot_table(
    merged_df,
    index='police_force',
    columns='vehicle_type',
    values='collision_index',
    aggfunc='count',
    fill_value=0
).round(2)

display(pivot_table)

vehicle_type,-1,1,2,3,4,5,8,9,10,11,...,17,19,20,21,22,23,90,97,98,99
police_force,,,,,,,,,,,,,,,,,,,,,
1,21,379,37,419,49,51,96,1734,4,109,...,0,177,7,22,3,2,12,11,8,0
3,0,3,0,3,1,3,2,95,0,1,...,2,9,2,5,0,0,0,1,0,0
4,1,28,0,8,0,5,0,326,0,1,...,1,20,0,5,0,0,2,1,5,0
5,3,31,1,5,3,7,6,175,0,5,...,1,14,0,1,1,2,1,2,0,0
6,3,29,1,5,2,5,10,264,0,3,...,0,17,1,3,0,0,1,1,2,0
7,0,15,2,7,1,6,2,164,0,3,...,2,15,1,14,0,1,1,0,0,0
10,0,28,0,7,0,4,4,168,0,14,...,1,16,0,5,2,2,0,3,1,0
11,0,8,0,1,2,1,0,52,1,0,...,0,2,0,2,0,1,0,1,1,0
12,0,28,1,3,2,8,1,130,0,1,...,0,9,1,3,1,0,0,0,0,0


### Observation for Step 9

The collision and vehicle datasets  merged using the collision_index column to combine collision-level and vehicle-level information. 
The grouped analysis by police_force provided information about driver age patterns, the spread of driver ages, the number of merged vehicle records, and the diversity of vehicle types within each police force area. 
The pivot table showed how vehicle records are distributed across different vehicle types and police force regions. 
This helps identify areas with larger numbers of specific vehicle categories and supports more detailed traffic and safety analysis.

## 10. Final Answer and Reproducibility

In [31]:
# final  summary table at collision level 

final_summary = clean_collision_df.groupby('police_force').agg(
    Avg_Multi_Vehicle_Collision=('multi_vehicle_collision','mean'),
    N_Collisions=('collision_index', 'nunique'),
    Total_Casualties=('number_of_casualties', 'sum')
    ).round(2).sort_values(
        by='Avg_Multi_Vehicle_Collision' ,
        ascending=False
    )
display(final_summary)

,Avg_Multi_Vehicle_Collision,N_Collisions,Total_Casualties
police_force,,,
43,0.80,246,291.0
7,0.79,120,150.0
34,0.79,112,140.0
52,0.78,193,252.0
40,0.77,101,150.0
41,0.77,163,218.0
54,0.76,98,136.0
45,0.74,200,253.0
17,0.74,42,68.0


In [32]:
# save final summary file to CSV

final_summary.reset_index().to_csv(
     "datasets/analysis_result_road_safety_by_force.csv",
     index=False
)

print("CSV file saved successfully")

CSV file saved successfully


### Final Conclusion

The final analysis summary identified the police force areas with the highest average multi-vehicle collision rates. 
The Avg_Multi_Vehicle_Collision column represents the average proportion of collisions involving multiple vehicles within each police force area. Higher values indicate that multi-vehicle collisions occur more frequently in those regions.Police_force 43 has highest Avg_Multi_vehicle_Collision of 0.80, meaning 80% of its collision involve multiple vehicles. 
The table also includes the total number of collisions and total casualties, which provide additional context for understanding the scale and severity of collisions across different police force areas. 
The summary table was successfully exported as `analysis_result_road_safety_by_force.csv` inside the datasets folder, ensuring that the notebook can regenerate the required output file when executed from top to bottom.